# 03 - Quality Checks

Objetivo: executar validações simples de qualidade sobre a camada Silver.

Essas validações serão usadas futuramente no README e na documentação de qualidade de dados.


In [ ]:
from pathlib import Path
import pandas as pd


In [ ]:
silver_base = Path("../data/silver/uf_alfabetizacao")

arquivos = sorted(silver_base.glob("execution_date=*/uf_alfabetizacao_silver.parquet"))

if not arquivos:
    raise FileNotFoundError("Nenhum arquivo Silver encontrado. Execute primeiro o notebook 02.")

silver_file = arquivos[-1]

print("Arquivo Silver usado:", silver_file)

df = pd.read_parquet(silver_file)
df.head()


In [ ]:
validacoes = {}

validacoes["qtd_linhas"] = len(df)
validacoes["qtd_colunas"] = len(df.columns)
validacoes["qtd_duplicados"] = int(df.duplicated().sum())
validacoes["ano_nulo"] = int(df["ano"].isna().sum()) if "ano" in df.columns else None
validacoes["sigla_uf_nulo"] = int(df["sigla_uf"].isna().sum()) if "sigla_uf" in df.columns else None
validacoes["taxa_alfabetizacao_nulo"] = int(df["taxa_alfabetizacao"].isna().sum()) if "taxa_alfabetizacao" in df.columns else None

validacoes


In [ ]:
# Regras de aprovação simples
erros = []

if validacoes["qtd_linhas"] == 0:
    erros.append("A base Silver está vazia.")

if validacoes["qtd_duplicados"] > 0:
    erros.append("Existem registros duplicados na base Silver.")

if validacoes["ano_nulo"] and validacoes["ano_nulo"] > 0:
    erros.append("Existem registros sem ano.")

if validacoes["sigla_uf_nulo"] and validacoes["sigla_uf_nulo"] > 0:
    erros.append("Existem registros sem sigla_uf.")

if erros:
    print("Validação reprovada:")
    for erro in erros:
        print("-", erro)
else:
    print("Validação aprovada com sucesso.")


In [ ]:
# Relatório simples de qualidade
relatorio = pd.DataFrame([
    {"validacao": chave, "resultado": valor}
    for chave, valor in validacoes.items()
])

relatorio
